In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from typing import Dict, List, Tuple, Optional

In [1]:
# Function to fetch stock data using yfinance
def fetch_stock_data(ticker, start_date, end_date):
    stock_data = yf.download(ticker, start=start_date, end=end_date, auto_adjust=True)
    return stock_data['Close']

# ticker : lot_size (for HK stocks)
tickers = {
    "XXX": 100,
    "YYY": 500,
    "AAA": 500,
    "BBB": 200,
}

# unique naming of the pairs
pairs = [ 
    "semi-conductor", 
    "automotives"
]

# z_score lookback window for each pair
windows = [
    10,
    5
]

# Define entry and exit thresholds for the z_scores
entry_thresholds = [
    2.0, 
    1.6
]
exit_thresholds = [
    0.4, 
    0.1
]

In [ ]:
start_date = "2020-01-01"
end_date = "2025-12-31"

In [ ]:
# Pairwise cointegration, correlation, and ADF spread test for all selected tickers

results = []

# Fetch price data for all tickers
price_data = fetch_stock_data(list(tickers.keys()), start_date, end_date)
price_data = price_data.ffill().dropna()
returns = (price_data.pct_change().dropna() + 1).cumprod() - 1

# Use the same series type as before (cumulative returns) for comparison
series = returns

for stock in tickers:
    paired_stock_data = returns[stock]
    # plt.plot(paired_stock_data, label=name_mapping[stock])
    plt.plot(paired_stock_data, label=stock)

plt.title('Stock Returns')
plt.xlabel('Date')
plt.xticks(rotation=45)
plt.ylabel('Return')
plt.legend()
plt.show()

In [ ]:
tick_list = list(tickers.keys())
left = price_data[tick_list[::2]].set_axis(pairs, axis=1)
right = price_data[tick_list[1::2]].set_axis(pairs, axis=1)
spreads = left - right

In [ ]:
# ----------------------------------------------------------------------
# Configuration and Constants
# ----------------------------------------------------------------------
class BacktestConfig:
    """Holds all configurable parameters for the backtest."""
    def __init__(self,
                 entry_thresholds: List[float],
                 exit_thresholds: List[float],
                 stop_loss_pct: float,
                 leverage: float,
                 maintenance_margin_pct: float,
                 borrowing_pct_cost_annual: float,
                 short_sell_pct_cost_annual: float,
                 initial_capital: float,
                 risk_free_rate_annual: float,
                 transaction_cost_pct: float = 0.1105 / 100,  # 0.1105% per side
                 fixed_transaction_cost: float = 15.0,
                 trade_priority_metric: str = 'excess_z'):  # or 'abs_z'
        self.entry_thresholds = entry_thresholds
        self.exit_thresholds = exit_thresholds
        self.stop_loss_pct = stop_loss_pct
        self.leverage = leverage
        self.maintenance_margin_pct = maintenance_margin_pct
        self.borrowing_pct_cost_annual = borrowing_pct_cost_annual
        self.short_sell_pct_cost_annual = short_sell_pct_cost_annual
        self.initial_capital = initial_capital
        self.risk_free_rate_annual = risk_free_rate_annual
        self.transaction_cost_pct = transaction_cost_pct
        self.fixed_transaction_cost = fixed_transaction_cost
        self.trade_priority_metric = trade_priority_metric


# ----------------------------------------------------------------------
# Position class to encapsulate a single pair's trade
# ----------------------------------------------------------------------
class Position:
    """Tracks a live position for one pair."""
    def __init__(self,
                 pair_name: str,
                 direction: int,          # 1 for long, -1 for short
                 a_shares: float,
                 b_shares: float,
                 entry_price1: float,
                 entry_price2: float,
                 entry_index: int,
                 entry_capital_used: float,
                 entry_cost: float):
        self.pair_name = pair_name
        self.direction = direction
        self.a_shares = a_shares
        self.b_shares = b_shares
        self.entry_price1 = entry_price1
        self.entry_price2 = entry_price2
        self.entry_index = entry_index
        self.entry_capital_used = entry_capital_used
        self.entry_cost = entry_cost
        self.current_pnl = 0.0

    def market_value(self, price1: float, price2: float) -> float:
        """Current absolute market value of the position."""
        return abs(self.a_shares * price1) + abs(self.b_shares * price2)

    def update_pnl(self, price1_prev: float, price2_prev: float,
                   price1_curr: float, price2_curr: float) -> float:
        """Calculate daily P&L change and update current_pnl."""
        if self.direction == 1:   # long pair: long A, short B
            daily_pnl = (self.a_shares * (price1_curr - price1_prev) -
                         self.b_shares * (price2_curr - price2_prev))
        else:                     # short pair: short A, long B
            daily_pnl = (self.b_shares * (price2_curr - price2_prev) -
                         self.a_shares * (price1_curr - price1_prev))
        self.current_pnl += daily_pnl
        return daily_pnl

    def financing_cost(self,
                       price1: float,
                       price2: float,
                       current_capital: float,
                       borrow_rate: float,
                       short_fee_rate: float) -> float:
        """Daily borrowing cost and short selling fee."""
        market_val = self.market_value(price1, price2)
        # cash borrowed = max(0, market_value - own_capital - accumulated PnL)
        own_capital = self.entry_capital_used + self.current_pnl
        borrowed = max(0.0, market_val - own_capital)
        borrow_cost = borrowed * (borrow_rate / 252.0)

        # short fee on the leg that is short
        if self.direction == 1:   # short B
            short_notional = self.b_shares * price2
        else:                     # short A
            short_notional = self.a_shares * price1
        short_fee = max(0.0, short_notional) * (short_fee_rate / 252.0)

        cost = -(borrow_cost + short_fee)
        # self.current_pnl += cost
        return cost


# ----------------------------------------------------------------------
# Main backtester class
# ----------------------------------------------------------------------
class MultiPairBacktester:
    """Runs a multi-pair statistical arbitrage backtest with dynamic capital allocation."""

    def __init__(self,
                 z_scores: pd.DataFrame,
                 price_data: Dict[str, pd.Series],
                 tickers: Dict[str, str],      # mapping ticker symbol -> name (if needed)
                 stock_lots: Dict[str, float],
                 pairs: List[str],
                 windows: List[int],
                 config: BacktestConfig):
        self.z_scores = z_scores
        self.price_data = price_data
        self.tickers = tickers
        self.stock_lots = stock_lots
        self.pairs = pairs
        self.windows = windows
        self.config = config

        # Derived: map pair name to its index in pairs list
        self.pair_to_idx = {pair: i for i, pair in enumerate(pairs)}

        # State variables
        self.active_positions: Dict[str, Position] = {}
        self.old_positions: List[str] = []
        self.current_capital = config.initial_capital
        self.daily_pct_returns: List[float] = []
        self.completed_trades: List[dict] = []
        self.margin_call_count = 0
        self.stop_loss_count = 0
        self.transaction_cost_today = 0.0

        # Pre‑compute signals for all pairs
        self.signals = self._generate_signals()

    def _generate_signals(self) -> pd.DataFrame:
        """Generate entry signals (+1 long, -1 short, 0 neutral) for each pair."""
        signals = pd.DataFrame(index=self.z_scores.index)
        for pair in self.pairs:
            idx = self.pair_to_idx[pair]
            entry = self.config.entry_thresholds[idx]
            signals[pair] = np.where(
                self.z_scores[pair] > entry, -1,
                np.where(self.z_scores[pair] < -entry, 1, 0)
            )
        return signals

    def run(self) -> Tuple[pd.DataFrame, List[float], pd.DataFrame]:
        """Execute the backtest loop."""
        for i in range(1, len(self.signals)):
            self._process_day(i)

        self._print_summary()
        self._plot_results()
        trades_df = pd.DataFrame(self.completed_trades)
        return trades_df, self.daily_pct_returns, self.signals

    # ------------------------------------------------------------------
    # Day processing
    # ------------------------------------------------------------------
    def _process_day(self, i: int):
        """Handle one day of the backtest: P&L update, exits, entries."""
        # 1. Update P&L for all active positions and compute total daily change
        daily_total = self._update_all_pnl(i)

        # 2. Update capital and record daily return
        if self.current_capital != 0:
            daily_return = daily_total / self.current_capital
        else:
            daily_return = 0.0
        self.daily_pct_returns.append(daily_return)
        self.current_capital += daily_total
        self.transaction_cost_today = 0.0  # reset after capital update

        # 3. Check exit conditions and close positions if needed
        self.old_positions = list(self.active_positions.keys())
        self._check_exits(i)

        # 4. Check entry signals for new positions
        self._check_entries(i)

    def _update_all_pnl(self, i: int) -> float:
        """Update P&L for all positions, including financing costs. Return total daily change."""
        total_daily = 0.0
        for pos in list(self.active_positions.values()):
            # Get prices for this pair
            stock1_ticker, stock2_ticker = self._get_pair_tickers(pos.pair_name)
            p1_prev = self.price_data[stock1_ticker].iloc[i-1]
            p2_prev = self.price_data[stock2_ticker].iloc[i-1]
            p1_curr = self.price_data[stock1_ticker].iloc[i]
            p2_curr = self.price_data[stock2_ticker].iloc[i]

            # P&L from price moves
            daily_pnl = pos.update_pnl(p1_prev, p2_prev, p1_curr, p2_curr)
            total_daily += daily_pnl

            # Financing costs
            fin_cost = pos.financing_cost(
                p1_curr, p2_curr,
                self.current_capital,
                self.config.borrowing_pct_cost_annual,
                self.config.short_sell_pct_cost_annual
            )
            total_daily += fin_cost

        # Add any transaction costs from previous day's closings/entries
        total_daily -= self.transaction_cost_today
        return total_daily

    def _check_exits(self, i: int):
        """Check each active position for exit triggers (stop‑loss, z‑score, margin)."""
        for pos in list(self.active_positions.values()):
            exit_reason = self._should_exit(pos, i)
            if exit_reason:
                self._close_position(pos, exit_reason, i)

    def _should_exit(self, pos: Position, i: int) -> Optional[str]:
        """Determine if a position should be closed. Returns reason string or None."""
        pair_idx = self.pair_to_idx[pos.pair_name]
        exit_threshold = self.config.exit_thresholds[pair_idx]

        # Stop‑loss based on percentage of current capital (original logic)
        pnl_pct = pos.current_pnl / self.current_capital
        if pnl_pct <= -self.config.stop_loss_pct:
            self.stop_loss_count += 1
            return 'stop_loss'

        # Z‑score mean reversion exit
        if abs(self.z_scores[pos.pair_name].iloc[i]) < exit_threshold:
            return 'z_score'

        # Margin call check
        stock1_ticker, stock2_ticker = self._get_pair_tickers(pos.pair_name)
        p1 = self.price_data[stock1_ticker].iloc[i]
        p2 = self.price_data[stock2_ticker].iloc[i]
        market_val = pos.market_value(p1, p2)
        if market_val > 0:
            margin_ratio = self.current_capital / market_val
            if margin_ratio < self.config.maintenance_margin_pct:
                self.margin_call_count += 1
                return 'margin_call'

        return None

    def _close_position(self, pos: Position, reason: str, i: int):
        """Close a position, record trade, and update transaction costs."""
        # Compute exit transaction cost (negative because it's a cost)
        exit_cost = self._transaction_cost(pos.entry_capital_used)

        # Record trade
        stock1_ticker, _ = self._get_pair_tickers(pos.pair_name)
        self.completed_trades.append({
            'pair': pos.pair_name,
            'entry_index': pos.entry_index,
            'entry_date': self.price_data[stock1_ticker].index[pos.entry_index],
            'exit_index': i,
            'exit_date': self.price_data[stock1_ticker].index[i],
            'position': pos.direction,
            'capital_used': pos.entry_capital_used,
            'entry_cost': pos.entry_cost,
            'pnl': pos.current_pnl,
            'exit_cost': exit_cost,
            'current_capital_at_exit': self.current_capital,
            'exit_reason': reason
        })

        # Accumulate transaction cost (will be applied next day)
        self.transaction_cost_today += exit_cost

        # Remove from active positions
        del self.active_positions[pos.pair_name]

    def _check_entries(self, i: int):
        """Look for entry signals and open new positions respecting available capital."""
        # Identify pairs with a signal today and not already active
        entry_candidates = {}
        for pair in self.pairs:
            if pair not in self.old_positions and self.signals[pair].iloc[i] != 0:
                entry_candidates[pair] = self.signals[pair].iloc[i]

        if not entry_candidates:
            return

        # Compute available capital
        used_capital = sum(p.entry_capital_used for p in self.active_positions.values())
        available_capital = self.current_capital - used_capital
        if available_capital <= 0:
            return

        # Prioritize candidates (highest excess z‑score)
        sorted_pairs = self._prioritize_entries(entry_candidates.keys(), i)

        # Process each candidate in order until capital runs out
        for pair in sorted_pairs:
            if available_capital <= 0:
                break
            signal_dir = entry_candidates[pair]
            cost = self._open_position_if_possible(pair, signal_dir, i, available_capital)
            if cost > 0:
                available_capital -= cost

    def _prioritize_entries(self, pair_names: List[str], i: int) -> List[str]:
        """Sort pairs by some metric; default: (|z| - entry)/entry."""
        def metric(pair):
            idx = self.pair_to_idx[pair]
            entry = self.config.entry_thresholds[idx]
            z_abs = abs(self.z_scores[pair].iloc[i])
            return (z_abs - entry) / entry
        return sorted(pair_names, key=metric, reverse=True)

    def _open_position_if_possible(self, pair: str, direction: int, i: int,
                                   available_capital: float) -> float:
        """
        Attempt to open a position for a given pair.
        Returns the amount of capital actually allocated (0 if cannot open).
        """
        idx = self.pair_to_idx[pair]
        stock1_ticker, stock2_ticker = self._get_pair_tickers(pair)

        p1 = self.price_data[stock1_ticker].iloc[i]
        p2 = self.price_data[stock2_ticker].iloc[i]
        lot1 = self.stock_lots[stock1_ticker]
        lot2 = self.stock_lots[stock2_ticker]

        # Determine base ratio
        if p2 * lot2 == 0:
            return 0.0
        price_ratio = (p1 * lot1) / (p2 * lot2)
        if price_ratio < 1:
            a_base = 1.0 / price_ratio
            b_base = 1.0
        else:
            a_base = 1.0
            b_base = price_ratio

        # Cost per unit (in base lots)
        cost_per_unit = a_base * lot1 * p1 + b_base * lot2 * p2
        if cost_per_unit <= 0:
            return 0.0

        # Determine multiple based on available capital and leverage
        max_allocation = available_capital  # can use all available (will be reduced after)
        multiple = int((max_allocation * self.config.leverage) / cost_per_unit)
        if multiple <= 0:
            return 0.0

        a_shares = int(a_base * multiple) * lot1
        b_shares = int(b_base * multiple) * lot2
        capital_used = a_shares * p1 + b_shares * p2
        entry_cost = self._transaction_cost(capital_used)  # negative because it's a cost

        # Create position
        pos = Position(
            pair_name=pair,
            direction=direction,
            a_shares=a_shares,
            b_shares=b_shares,
            entry_price1=p1,
            entry_price2=p2,
            entry_index=i,
            entry_capital_used=capital_used,
            entry_cost=entry_cost
        )
        self.active_positions[pair] = pos
        self.transaction_cost_today += entry_cost

        return capital_used

    # ------------------------------------------------------------------
    # Helpers
    # ------------------------------------------------------------------
    def _get_pair_tickers(self, pair: str) -> Tuple[str, str]:
        """Return the two ticker symbols for a given pair name."""
        idx = self.pair_to_idx[pair]
        ticker_list = list(self.tickers.keys())
        return ticker_list[idx * 2], ticker_list[idx * 2 + 1]

    def _transaction_cost(self, notional: float) -> float:
        """Calculate total transaction cost (commission + fixed)."""
        return notional * self.config.transaction_cost_pct + self.config.fixed_transaction_cost

    # ------------------------------------------------------------------
    # Reporting
    # ------------------------------------------------------------------
    def _print_summary(self):
        """Print performance metrics."""
        trades = self.completed_trades
        n_trades = len(trades)
        print(f"Completed trades: {n_trades}")
        print(f"Stop losses triggered: {self.stop_loss_count}")

        if n_trades > 0:
            trades_df = pd.DataFrame(trades)
            # Net profit per trade = pnl + entry_cost + exit_cost
            trades_df['net_profit'] = trades_df['pnl'] + trades_df['entry_cost'] + trades_df['exit_cost']
            win_rate = (trades_df['net_profit'] > 0).mean()
            print(f"Win Rate: {win_rate:.2%}")

        print(f"Margin calls: {self.margin_call_count}")
        total_return = (self.current_capital - self.config.initial_capital) / self.config.initial_capital
        print(f"Total return: {total_return:.2%}")

        if self.daily_pct_returns:
            ann_return = (self.current_capital / self.config.initial_capital) ** (252 / len(self.daily_pct_returns)) - 1
            print(f"Annualized return: {ann_return * 100:.2f}%")

        # Sharpe ratio
        daily_ret = np.array(self.daily_pct_returns)
        daily_rf = (1 + self.config.risk_free_rate_annual) ** (1/252) - 1
        ex_ret = daily_ret - daily_rf
        mean_ex = np.nanmean(ex_ret)
        std_ex = np.nanstd(ex_ret)
        sharpe = (mean_ex / std_ex) * np.sqrt(252) if std_ex > 0 else np.nan
        print(f"Sharpe Ratio (annualized): {sharpe:.4f}")

    def _plot_results(self):
        """Plot cumulative returns of the strategy."""
        if not self.daily_pct_returns:
            return
        cumulative = np.cumprod(1 + np.array(self.daily_pct_returns)) - 1
        # Find the index range corresponding to days 1..end
        # Use the first ticker's index for dates (they all share same index)
        first_ticker = list(self.price_data.keys())[0]
        dates = self.price_data[first_ticker].index[1:1+len(cumulative)]

        fig, ax = plt.subplots(figsize=(12, 6))
        ax.plot(dates, cumulative, label='Multi-Pair Strategy', linewidth=2)
        ax.set_title('Multi-Pair Trading Strategy Cumulative Returns')
        ax.set_xlabel('Date')
        ax.set_ylabel('Cumulative Returns')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()


# ----------------------------------------------------------------------
# Convenience wrapper (backward‑compatible call)
# ----------------------------------------------------------------------
def backtest_strategy_multi_pair(z_scores, price_data, tickers, stock_lots, pairs, windows,
                                 leverage, maintenance_margin_pct, borrowing_pct_cost_annual,
                                 short_sell_pct_cost_annual, entry_thresholds, exit_thresholds,
                                 stop_loss_pct, initial_capital, risk_free_rate_annual,):
    """
    Legacy wrapper for the refactored backtester.
    """
    config = BacktestConfig(
        entry_thresholds=entry_thresholds,
        exit_thresholds=exit_thresholds,
        stop_loss_pct=stop_loss_pct,
        leverage=leverage,
        maintenance_margin_pct=maintenance_margin_pct,
        borrowing_pct_cost_annual=borrowing_pct_cost_annual,
        short_sell_pct_cost_annual=short_sell_pct_cost_annual,
        initial_capital=initial_capital,
        risk_free_rate_annual=risk_free_rate_annual
    )
    backtester = MultiPairBacktester(
        z_scores=z_scores,
        price_data=price_data,
        tickers=tickers,
        stock_lots=stock_lots,
        pairs=pairs,
        windows=windows,
        config=config
    )
    return backtester.run()

In [ ]:
# Calculate z-score of the spread based on a rolling window
z_scores = pd.DataFrame(columns=pairs)
for i in range(len(windows)):
    spread_mean = spreads[pairs[i]].rolling(window=windows[i]).mean()
    spread_std = spreads[pairs[i]].rolling(window=windows[i]).std()
    z_scores[pairs[i]] = (spreads[pairs[i]] - spread_mean) / spread_std

In [ ]:
# Risk params: leverage and maintenance margin for margin call
leverage = 1
maintenance_margin_pct = 0.3
borrowing_pct_cost_annual = 0.068
short_sell_pct_cost_annual = 0.0112

stop_loss_pct = 0.01 * leverage ** 2

# Define initial capital
initial_capital = 100000 * len(pairs)

# Sharpe ratio parameters
risk_free_rate_annual = 0.03665
trades_log, pct_returns, signals = backtest_strategy_multi_pair(
    z_scores=z_scores,
    price_data=price_data,
    tickers=tickers,
    stock_lots=tickers,  # dict mapping ticker to shares per lot
    pairs=pairs,
    windows=windows,
    leverage=leverage,
    maintenance_margin_pct=maintenance_margin_pct,
    borrowing_pct_cost_annual=borrowing_pct_cost_annual,
    short_sell_pct_cost_annual=short_sell_pct_cost_annual,
    entry_thresholds=entry_thresholds,
    exit_thresholds=exit_thresholds,
    stop_loss_pct=stop_loss_pct,
    initial_capital=initial_capital,
    risk_free_rate_annual=risk_free_rate_annual,
)

In [ ]:
# Save/print trade log
if not trades_log.empty:
    pass
    display(trades_log)
    # display(trades_log[pd.to_datetime(trades_log['entry_date']).dt.strftime("%Y") == '2023'])
    # optional: save to CSV
    # trades_log.to_csv('trade_log.csv', index=False)

# Alpha Analysis

### Daily

In [ ]:
factor_df = pd.read_csv('F-F_Research_Data_Factors_daily.csv', skiprows=4, header=0)
mom_df = pd.read_csv('F-F_Momentum_Factor_daily.csv', skiprows=13, header=0)

# parse Date (files use YYYYMMDD) and coerce invalid/footer rows
factor_df['Date'] = pd.to_datetime(factor_df['Date'], format='%Y%m%d', errors='coerce')
mom_df['Date'] = pd.to_datetime(mom_df['Date'], format='%Y%m%d', errors='coerce')

# # drop rows with invalid dates (footers)
# factor_df = factor_df.dropna(subset=['Date']).copy()
# mom_df = mom_df.dropna(subset=['Date']).copy()

# remove any unnamed/footer columns from momentum file
mom_df = mom_df.loc[:, ~mom_df.columns.str.contains('^Unnamed')]

# merge, set Date as index and align to price_data index
carhart_df = pd.merge(factor_df, mom_df, on='Date', how='inner')
carhart_df = carhart_df.set_index('Date').sort_index()
carhart_df = carhart_df.loc[carhart_df.index.isin(price_data.index)]

# ensure numeric and convert from percent to decimal
carhart_df = carhart_df.apply(pd.to_numeric, errors='coerce') / 100
carhart_df = carhart_df[1:]
carhart_df

In [ ]:
# convert pct_returns (list) to a pandas Series with proper dates, then align with Carhart factors
strategy_returns = pd.Series(pct_returns, index=price_data.index[1:1+len(pct_returns)])

# align indices
common_idx = strategy_returns.index.intersection(carhart_df.index)

y = strategy_returns.loc[common_idx] - carhart_df.loc[common_idx, 'RF']
X = sm.add_constant(carhart_df.loc[common_idx, ['Mkt-RF', 'SMB', 'HML', 'Mom']])

reg = sm.OLS(y, X).fit()
print("annaulized alpha", (1+reg.params.iloc[0])**252-1)
print(reg.summary())

### Monthly

In [ ]:
factor_df = pd.read_excel('Carhart_Monthly.xlsx', sheet_name='Monthly_Carhart', header=0)
mom_df = pd.read_excel('Carhart_Monthly.xlsx', sheet_name='Monthly_MOM', header=0)

# parse Date (files use YYYYMMDD) and coerce invalid/footer rows
factor_df['Date'] = pd.to_datetime(factor_df['Date'], format='%Y%m', errors='coerce').dt.strftime('%Y-%m')
mom_df['Date'] = pd.to_datetime(mom_df['Date'], format='%Y%m', errors='coerce').dt.strftime('%Y-%m')

# merge, set Date as index and align to price_data index
carhart_df = pd.merge(factor_df, mom_df, on='Date', how='inner')
carhart_df = carhart_df.set_index('Date').sort_index()
carhart_df = carhart_df.loc[carhart_df.index.isin(price_data.index.strftime('%Y-%m'))]

# ensure numeric and convert from percent to decimal
carhart_df = carhart_df.apply(pd.to_numeric, errors='coerce') / 100

In [ ]:
# convert pct_returns (list) to a pandas Series with proper dates, then align with Carhart factors
strategy_returns = pd.Series(pct_returns, index=price_data.index[1:1+len(pct_returns)])
monthly_returns = (1 + strategy_returns).resample('ME').prod() - 1
monthly_returns.index = monthly_returns.index.strftime('%Y-%m')

# align indices
common_idx = monthly_returns.index.intersection(carhart_df.index)

y = monthly_returns.loc[common_idx] - carhart_df.loc[common_idx, 'RF']
X = sm.add_constant(carhart_df.loc[common_idx, ['Mkt-RF', 'SMB', 'HML', 'Mom']])

reg = sm.OLS(y, X).fit()
print("annaulized alpha", (1+reg.params.iloc[0])**12-1)
print(reg.summary())